# Q3A — Sonic Signatures: 'Magical Mystery Tune'
Full Shazam-style audio fingerprinting pipeline, run cell by cell in Google Colab.

## Step 1 — Install dependencies

In [ ]:
!pip install -q librosa

## Step 2 — Upload your song database
Run this, then pick your song-database `.zip` file when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()   # choose your song database .zip


In [ ]:
import zipfile, os
zip_name = list(uploaded.keys())[0]
os.makedirs('/content/song_db', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/song_db')
print('Extracted to /content/song_db')
!find /content/song_db -maxdepth 3 -type d


## Step 3 — Set the database path
Edit `DB_DIR` below if your folder name/path differs from what was printed above.

In [ ]:
DB_DIR = '/content/song_db/EE200 Project Song Database'   # <-- EDIT IF NEEDED
OUT    = '/content/figs'
import os
os.makedirs(OUT, exist_ok=True)
print('Songs found:', len([f for f in os.listdir(DB_DIR) if f.endswith('.mp3')]))


## Step 4 — Imports and plotting helpers

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.signal import spectrogram as sp_spectrogram
from scipy.ndimage import maximum_filter
import librosa
import pickle, collections, warnings
warnings.filterwarnings('ignore')

DARK='#0d1117'; PANEL='#111827'; GREEN='#4ade80'; BLUE='#60a5fa'
YELLOW='#facc15'; RED='#f87171'; PURPLE='#a78bfa'; GRAY='#9ca3af'

def style(ax):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=GRAY, labelsize=8)
    for sp in ax.spines.values(): sp.set_edgecolor('#374151')
    ax.xaxis.label.set_color(GRAY); ax.yaxis.label.set_color(GRAY)

def savefig(fig, name):
    fig.savefig(f'{OUT}/{name}.png', dpi=150, bbox_inches='tight', facecolor=DARK)
    plt.show()
    plt.close(fig)
    print(f'  saved {name}.png')


## Step 5 — Audio loading

In [ ]:
SR = 8000   # downsample to 8kHz (sufficient for 0-4kHz music content)

def load_audio(path, sr=SR, duration=None, offset=0.0):
    y, _ = librosa.load(path, sr=sr, mono=True, duration=duration, offset=offset)
    return y

songs = sorted([f for f in os.listdir(DB_DIR) if f.endswith('.mp3')])
print(f'Found {len(songs)} songs in database')

DEMO_SONG = 'Let It Be.mp3' if 'Let It Be.mp3' in songs else songs[0]
demo_path = os.path.join(DB_DIR, DEMO_SONG)
y_demo30  = load_audio(demo_path, duration=30)
y_demo_full = load_audio(demo_path)
print(f'Demo song: {DEMO_SONG}  ({len(y_demo_full)/SR:.1f}s @ {SR}Hz)')


## Step 6 — Single full-song FFT
Shows *which* frequencies exist in the whole song, but all timing information is lost — motivates the spectrogram.

In [ ]:
Y = np.fft.rfft(y_demo_full)
freqs = np.fft.rfftfreq(len(y_demo_full), d=1/SR)
mag = np.abs(Y)

fig, ax = plt.subplots(figsize=(12,4.2), facecolor=DARK); style(ax)
ax.plot(freqs, mag, color=BLUE, lw=0.6)
ax.set_xlim(0, 4000)
ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('Magnitude')
ax.set_title(f'Single DFT of entire song -- "{DEMO_SONG[:-4]}" ({len(y_demo_full)/SR:.0f}s)\n'
             'Shows WHICH frequencies are present, but WHEN they occur is completely lost',
             color='white', fontsize=11)
plt.tight_layout()
savefig(fig, 'q3a_0_full_song_fft')


## Step 7 — Spectrogram: short vs long vs optimal window
Compares time/frequency resolution tradeoffs.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 13), facecolor=DARK)
for ax in axes: style(ax)

configs = [
    (256,   'Short window (256 samp / 32ms)\nGood TIME resolution, poor FREQ resolution'),
    (2048,  'Long window (2048 samp / 256ms)\nGood FREQ resolution, poor TIME resolution'),
    (1024,  'Optimal window (1024 samp / 128ms)\nBalanced time-frequency resolution  <- used for fingerprinting'),
]

for ax, (nperseg, title) in zip(axes, configs):
    f, t, Sxx = sp_spectrogram(y_demo30, fs=SR, nperseg=nperseg, noverlap=nperseg*3//4, scaling='spectrum')
    im = ax.pcolormesh(t, f, 10*np.log10(Sxx + 1e-10), shading='gouraud', cmap='inferno', vmin=-80, vmax=0)
    ax.set_ylim(0, 4000)
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Frequency (Hz)')
    ax.set_title(title, color='white', fontsize=11, pad=6)
    plt.colorbar(im, ax=ax, label='dB', pad=0.01)

fig.suptitle(f'Spectrogram Comparison -- Short vs Long Window\n(Demo: "{DEMO_SONG[:-4]}")',
             color=BLUE, fontsize=14, fontweight='bold')
plt.tight_layout()
savefig(fig, 'q3a_1_spectrogram_windows')


## Step 8 — Fingerprinting core functions
Spectrogram -> constellation (local-max peaks) -> paired hashes.

In [ ]:
NPERSEG  = 1024
NOVERLAP = 768
N_PEAKS  = 15
FAN_OUT  = 5
DT_MIN   = 1
DT_MAX   = 100

def compute_spectrogram(y, sr=SR):
    f, t, Sxx = sp_spectrogram(y, fs=sr, nperseg=NPERSEG, noverlap=NOVERLAP, scaling='spectrum')
    return f, t, np.log10(Sxx + 1e-10)

def get_constellation(f, t, S, n_peaks=N_PEAKS):
    local_max = maximum_filter(S, size=(10, 10)) == S
    threshold  = S.mean() + 0.8 * S.std()
    strong     = S > threshold
    peaks_mask = local_max & strong
    peak_rows, peak_cols = np.where(peaks_mask)
    peaks_by_t = collections.defaultdict(list)
    for r, c in zip(peak_rows, peak_cols):
        peaks_by_t[c].append((S[r, c], r, c))
    constellation = []
    for c, pts in peaks_by_t.items():
        pts.sort(reverse=True)
        for _, r, col in pts[:n_peaks]:
            constellation.append((r, col))
    return sorted(constellation, key=lambda x: x[1])

def make_hashes(constellation, fan_out=FAN_OUT, dt_min=DT_MIN, dt_max=DT_MAX):
    hashes = []
    pts = sorted(constellation, key=lambda x: x[1])
    for i, (f1, t1) in enumerate(pts):
        for j in range(i+1, min(i+1+fan_out, len(pts))):
            f2, t2 = pts[j]
            dt = t2 - t1
            if dt_min <= dt <= dt_max:
                hashes.append(((int(f1), int(f2), int(dt)), t1))
    return hashes


## Step 9 — Build the fingerprint database (indexes every song)
This may take a minute or two depending on database size.

In [ ]:
database   = {}
song_names = []

for i, song_file in enumerate(songs):
    name = song_file.replace('.mp3', '')
    path = os.path.join(DB_DIR, song_file)
    y    = load_audio(path)
    f, t, S = compute_spectrogram(y)
    constellation = get_constellation(f, t, S)
    hashes = make_hashes(constellation)
    for h, t_anc in hashes:
        database.setdefault(h, []).append((name, t_anc))
    song_names.append(name)
    print(f'  [{i+1:2d}/{len(songs)}] {name[:40]:<40} {len(hashes):6d} hashes')

print(f'\nDatabase: {len(database):,} unique hashes across {len(songs)} songs')

with open('/content/fingerprint_db.pkl', 'wb') as fp:
    pickle.dump({'database': database, 'song_names': song_names}, fp)
print('Database saved -> /content/fingerprint_db.pkl')


## Step 10 — Matching function
Looks up a query clip's hashes against the database and scores songs by their largest offset-histogram peak.

In [ ]:
def identify_clip(y, database, song_names=None, use_pairs=True):
    f, t, S = compute_spectrogram(y)
    const   = get_constellation(f, t, S)
    if use_pairs:
        q_hashes = make_hashes(const)
    else:
        q_hashes = [((int(fi),), ti) for fi, ti in const]

    offsets = collections.defaultdict(lambda: collections.defaultdict(int))
    for h, t_q in q_hashes:
        if h in database:
            for (song, t_db) in database[h]:
                delta = int(t_db) - int(t_q)
                offsets[song][delta] += 1

    if not offsets:
        return None, {}, const, S, f, t

    scores = {song: max(hist.values()) for song, hist in offsets.items()}
    matched = max(scores, key=scores.get)
    return matched, scores, const, S, f, t


## Step 11 — Visualize constellation + paired hashes on the demo song

In [ ]:
f_arr, t_arr, S = compute_spectrogram(y_demo30)
constellation    = get_constellation(f_arr, t_arr, S)
hashes           = make_hashes(constellation)

fig, axes = plt.subplots(3, 1, figsize=(14, 14), facecolor=DARK)
for ax in axes: style(ax)

im = axes[0].pcolormesh(t_arr, f_arr, 10*S, shading='gouraud', cmap='inferno', vmin=-80, vmax=0)
axes[0].set_ylim(0, 4000)
axes[0].set_title(f'(a) Spectrogram -- "{DEMO_SONG[:-4]}"', color='white', fontsize=12)
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('Frequency (Hz)')
plt.colorbar(im, ax=axes[0], label='dB')

axes[1].pcolormesh(t_arr, f_arr, 10*S, shading='gouraud', cmap='inferno', vmin=-80, vmax=0, alpha=0.4)
peaks_t = [t_arr[c] for _, c in constellation]
peaks_f = [f_arr[r] for r, _ in constellation]
axes[1].scatter(peaks_t, peaks_f, color=YELLOW, s=6, alpha=0.9, linewidths=0)
axes[1].set_ylim(0, 4000)
axes[1].set_title(f'(b) Constellation Map -- {len(constellation)} peaks (local maxima)', color='white', fontsize=12)
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Frequency (Hz)')

axes[2].pcolormesh(t_arr, f_arr, 10*S, shading='gouraud', cmap='inferno', vmin=-80, vmax=0, alpha=0.3)
axes[2].scatter(peaks_t, peaks_f, color=YELLOW, s=4, alpha=0.6, linewidths=0)
shown = 0
for (f1, f2, dt), t_anc in hashes[:200]:
    t1 = t_arr[min(t_anc, len(t_arr)-1)]
    t2 = t1 + dt * (NPERSEG - NOVERLAP) / SR
    axes[2].plot([t1, t2], [f_arr[f1], f_arr[f2]], color=GREEN, lw=0.5, alpha=0.5)
    shown += 1
axes[2].set_ylim(0, 4000)
axes[2].set_title(f'(c) Paired Hashes -- {len(hashes):,} total (first {shown} shown)', color='white', fontsize=12)
axes[2].set_xlabel('Time (s)'); axes[2].set_ylabel('Frequency (Hz)')

fig.suptitle('Spectrogram -> Constellation -> Hash Pairs', color=BLUE, fontsize=14, fontweight='bold')
plt.tight_layout()
savefig(fig, 'q3a_2_constellation_hashes')


## Step 12 — Query a clip and visualize the offset histogram
A correct match shows hashes lining up at a single sharp offset; a wrong song shows scattered offsets.

In [ ]:
QUERY_SONG = 'Yesterday.mp3' if 'Yesterday.mp3' in songs else songs[1]
y_query = load_audio(os.path.join(DB_DIR, QUERY_SONG), duration=10, offset=15)
matched, scores, q_const, q_S, q_f, q_t = identify_clip(y_query, database, song_names)
print(f'  Query: {QUERY_SONG}   ->   Match: {matched}')

top5 = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:5]
print(f'  Top 5: {top5}')

q_hashes = make_hashes(get_constellation(q_f, q_t, q_S))
offsets_matched = collections.defaultdict(int)
for h, t_q in q_hashes:
    if h in database:
        for (song, t_db) in database[h]:
            if song == matched:
                offsets_matched[int(t_db) - int(t_q)] += 1

fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor=DARK)
for ax in axes.flat: style(ax)

im = axes[0,0].pcolormesh(q_t, q_f, 10*q_S, shading='gouraud', cmap='inferno', vmin=-80, vmax=0)
axes[0,0].set_ylim(0, 4000)
qpt = [q_t[c] for _, c in q_const]; qpf = [q_f[r] for r, _ in q_const]
axes[0,0].scatter(qpt, qpf, color=YELLOW, s=8, alpha=0.9, linewidths=0)
axes[0,0].set_title(f'Query clip spectrogram\n"{QUERY_SONG[:-4]}" (10s)', color='white', fontsize=11)
axes[0,0].set_xlabel('Time (s)'); axes[0,0].set_ylabel('Frequency (Hz)')
plt.colorbar(im, ax=axes[0,0], label='dB')

deltas_m = sorted(offsets_matched.keys())
counts_m = [offsets_matched[d] for d in deltas_m]
axes[0,1].bar(deltas_m, counts_m, color=GREEN, alpha=0.85, width=1)
peak_offset = max(offsets_matched, key=offsets_matched.get)
axes[0,1].axvline(peak_offset, color=RED, lw=2, linestyle='--', label=f'Peak offset={peak_offset} (true match!)')
axes[0,1].set_xlabel('Time offset (frames)'); axes[0,1].set_ylabel('Hash matches')
axes[0,1].set_title(f'Offset histogram -- matched song\n"{matched}"\nPeak = {offsets_matched[peak_offset]} matches', color='white', fontsize=11)
axes[0,1].legend(fontsize=8, facecolor='#1f2937', labelcolor='white')

wrong_song = [s for s in song_names if s != matched][3]
offsets_wrong = collections.defaultdict(int)
for h, t_q in q_hashes:
    if h in database:
        for (song, t_db) in database[h]:
            if song == wrong_song:
                offsets_wrong[int(t_db) - int(t_q)] += 1
dw = sorted(offsets_wrong.keys()); cw = [offsets_wrong[d] for d in dw]
axes[1,0].bar(dw, cw, color=RED, alpha=0.7, width=1)
axes[1,0].set_xlabel('Time offset (frames)'); axes[1,0].set_ylabel('Hash matches')
axes[1,0].set_title(f'Offset histogram -- WRONG song\n"{wrong_song}"\nScattered (no clear peak)', color='white', fontsize=11)

names5  = [s[:20] for s, _ in top5]
scores5 = [sc for _, sc in top5]
cols5   = [GREEN] + [RED]*(len(top5)-1)
bars    = axes[1,1].barh(names5[::-1], scores5[::-1], color=cols5[::-1], alpha=0.85, edgecolor='white', lw=0.5)
axes[1,1].set_xlabel('Peak offset matches'); axes[1,1].set_title('Top-5 Song Scores', color='white', fontsize=11)
for bar, v in zip(bars[::-1], scores5):
    axes[1,1].text(v+0.5, bar.get_y()+bar.get_height()/2, str(v), va='center', color='white', fontsize=9, fontweight='bold')

fig.suptitle(f'Query Matching: Correct="{matched}"  (Query="{QUERY_SONG[:-4]}")', color=BLUE, fontsize=13, fontweight='bold')
plt.tight_layout()
savefig(fig, 'q3a_3_matching_histogram')


## Step 13 — Single peaks vs paired hashes
Shows why pairing two peaks into one hash is far more decisive than matching on single peaks alone.

In [ ]:
db_single = {}
for song_file in songs:
    name = song_file.replace('.mp3', '')
    path = os.path.join(DB_DIR, song_file)
    y    = load_audio(path)
    f, t, S = compute_spectrogram(y)
    const = get_constellation(f, t, S)
    for fi, ti in const:
        db_single.setdefault((int(fi),), []).append((name, ti))

def match_single(y):
    f, t, S = compute_spectrogram(y)
    const   = get_constellation(f, t, S)
    offsets = collections.defaultdict(lambda: collections.defaultdict(int))
    for fi, ti in const:
        h = (int(fi),)
        if h in db_single:
            for (song, t_db) in db_single[h]:
                offsets[song][int(t_db)-int(ti)] += 1
    if not offsets:
        return None, {}
    scores = {song: max(hist.values()) for song, hist in offsets.items()}
    return max(scores, key=scores.get), scores

matched_pair,   scores_pair   = matched, scores
matched_single, scores_single = match_single(y_query)
print(f'  Paired hashes -> {matched_pair}')
print(f'  Single peaks  -> {matched_single}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=DARK)
for ax in axes: style(ax)

offsets_pair = offsets_matched
dp = sorted(offsets_pair.keys()); cp = [offsets_pair[d] for d in dp]
axes[0].bar(dp, cp, color=GREEN, alpha=0.85, width=1)
pk = max(offsets_pair, key=offsets_pair.get)
axes[0].axvline(pk, color=YELLOW, lw=2.5, linestyle='--', label=f'Peak={offsets_pair[pk]} matches')
axes[0].set_xlabel('Offset (frames)'); axes[0].set_ylabel('Count')
axes[0].set_title(f'PAIRED hashes -> "{matched_pair[:25]}"\nSharp single peak -- decisive!', color='white', fontsize=11)
axes[0].legend(fontsize=9, facecolor='#1f2937', labelcolor='white')

if matched_single:
    q_const_s = get_constellation(q_f, q_t, q_S)
    off_s = collections.defaultdict(int)
    for fi, ti in q_const_s:
        h = (int(fi),)
        if h in db_single:
            for (song, t_db) in db_single[h]:
                if song == matched_single:
                    off_s[int(t_db)-int(ti)] += 1
    ds = sorted(off_s.keys()); cs = [off_s[d] for d in ds]
    axes[1].bar(ds, cs, color=RED, alpha=0.7, width=1)
    if off_s:
        pk_s = max(off_s, key=off_s.get)
        axes[1].axvline(pk_s, color=YELLOW, lw=2, linestyle='--', label=f'Peak={off_s[pk_s]} matches')
        axes[1].legend(fontsize=9, facecolor='#1f2937', labelcolor='white')
axes[1].set_xlabel('Offset (frames)'); axes[1].set_ylabel('Count')
axes[1].set_title(f'SINGLE peaks -> "{(matched_single or "?")[:25]}"\nFlat/noisy -- many false matches', color='white', fontsize=11)

fig.suptitle('Single Peaks vs Paired Hashes: Why Pairing is Decisive', color=BLUE, fontsize=14, fontweight='bold')
plt.tight_layout()
savefig(fig, 'q3a_4_single_vs_paired')


## Step 14 — Noise robustness test
Add increasing amounts of noise to the query clip and see how far recognition holds up.

In [ ]:
noise_levels = [0, 0.01, 0.05, 0.10, 0.20, 0.40]
noise_results = []
for snr in noise_levels:
    y_noisy = y_query + snr * np.random.randn(len(y_query)) * np.std(y_query)
    m, sc, _, _, _, _ = identify_clip(y_noisy, database, song_names)
    correct = (m == QUERY_SONG.replace('.mp3',''))
    peak    = sc.get(m, 0) if sc else 0
    noise_results.append((snr, m, correct, peak))
    print(f'  noise={snr:.2f} -> {m}  {"OK" if correct else "FAIL"}  peak={peak}')

fig, axes = plt.subplots(2, 3, figsize=(16, 9), facecolor=DARK)
for ax in axes.flat: style(ax)

for ax, (snr, m, correct, peak) in zip(axes.flat, noise_results):
    y_noisy = y_query + snr * np.random.randn(len(y_query)) * np.std(y_query)
    fn, tn, Sn = compute_spectrogram(y_noisy)
    ax.pcolormesh(tn, fn, 10*Sn, shading='gouraud', cmap='inferno', vmin=-80, vmax=0)
    ax.set_ylim(0, 4000)
    col   = GREEN if correct else RED
    mark  = 'OK' if correct else 'FAIL'
    ax.set_title(f'Noise sigma={snr:.2f} | {mark} "{(m or "?")[:20]}"\nPeak matches={peak}', color=col, fontsize=9, pad=5)
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Freq (Hz)')

fig.suptitle('Noise Robustness: How Much Noise Before Recognition Fails?', color=BLUE, fontsize=14, fontweight='bold')
plt.tight_layout()
savefig(fig, 'q3a_5_noise_robustness')


## Step 15 — Pitch-shift robustness test
Shift the query clip's pitch up/down and see why even small shifts defeat the identifier.

In [ ]:
pitch_shifts = [-4, -2, -1, 0, 1, 2, 4]
pitch_results = []
for n_steps in pitch_shifts:
    if n_steps == 0:
        y_shifted = y_query.copy()
    else:
        y_shifted = librosa.effects.pitch_shift(y_query, sr=SR, n_steps=n_steps)
    m, sc, _, _, _, _ = identify_clip(y_shifted, database, song_names)
    correct = (m == QUERY_SONG.replace('.mp3',''))
    peak    = sc.get(m, 0) if sc else 0
    pitch_results.append((n_steps, m, correct, peak))
    print(f'  pitch={n_steps:+d} semitones -> {m}  {"OK" if correct else "FAIL"}  peak={peak}')

fig, axes = plt.subplots(2, 4, figsize=(18, 9), facecolor=DARK)
axes_flat = list(axes.flat)

for (n_steps, m, correct, peak), ax in zip(pitch_results, axes_flat):
    style(ax)
    if n_steps == 0:
        y_sh = y_query.copy()
    else:
        y_sh = librosa.effects.pitch_shift(y_query, sr=SR, n_steps=n_steps)
    fsh, tsh, Ssh = compute_spectrogram(y_sh)
    ax.pcolormesh(tsh, fsh, 10*Ssh, shading='gouraud', cmap='plasma', vmin=-80, vmax=0)
    ax.set_ylim(0, 4000)
    col  = GREEN if correct else RED
    mark = 'OK' if correct else 'FAIL'
    ax.set_title(f'Shift={n_steps:+d} semitones | {mark}\n"{(m or "?")[:18]}"  peak={peak}', color=col, fontsize=9, pad=5)
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Freq (Hz)')

axes_flat[len(pitch_results)].axis('off')

fig.suptitle('Pitch-Shift Robustness: Why Small Shifts Defeat the Identifier\n'
             '(Peaks shift in frequency -> different freq_bins -> no hash matches)',
             color=BLUE, fontsize=14, fontweight='bold')
plt.tight_layout()
savefig(fig, 'q3a_6_pitch_robustness')


## Step 15b — Time-stretch robustness test
Stretch/compress the query clip's tempo (pitch-preserving) and see how confidence degrades — contrast this with the pitch-shift failure above.

In [ ]:
def time_stretch(y, rate):
    """Pitch-preserving time stretch using librosa's phase vocoder.
    rate > 1.0 = faster/shorter, rate < 1.0 = slower/longer."""
    return librosa.effects.time_stretch(y, rate=rate)

stretch_rates = [0.90, 0.95, 0.98, 1.00, 1.02, 1.05, 1.10]
stretch_results = []
for rate in stretch_rates:
    y_s = y_query.copy() if rate == 1.0 else time_stretch(y_query, rate)
    m, sc, _, _, _, _ = identify_clip(y_s, database, song_names)
    correct = (m == QUERY_SONG.replace('.mp3',''))
    peak    = sc.get(m, 0) if sc else 0
    stretch_results.append((rate, m, correct, peak))
    print(f'  stretch_rate={rate:.2f} -> {m}  {"OK" if correct else "FAIL"}  peak={peak}')

fig, axes = plt.subplots(2, 4, figsize=(18, 9), facecolor=DARK)
axes_flat = list(axes.flat)

for (rate, m, correct, peak), ax in zip(stretch_results, axes_flat):
    style(ax)
    y_s = y_query.copy() if rate == 1.0 else time_stretch(y_query, rate)
    fst, tst, Sst = compute_spectrogram(y_s)
    ax.pcolormesh(tst, fst, 10*Sst, shading='gouraud', cmap='viridis', vmin=-80, vmax=0)
    ax.set_ylim(0, 4000)
    col  = GREEN if correct else RED
    mark = 'OK' if correct else 'FAIL'
    pct  = (rate - 1.0) * 100
    ax.set_title(f'Stretch={pct:+.0f}% (rate={rate:.2f}) | {mark}\n"{(m or "?")[:18]}"  peak={peak}',
                 color=col, fontsize=9, pad=5)
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Freq (Hz)')
axes_flat[len(stretch_results)].axis('off')

fig.suptitle('Time-Stretch Robustness: Tempo Changes Shift Peaks in TIME, Not Frequency\n'
             '(Recognition survives, but confidence drops quickly away from original tempo)',
             color=BLUE, fontsize=13, fontweight='bold')
plt.tight_layout()
savefig(fig, 'q3a_8_timestretch_robustness')


## Step 16 — Full database accuracy test
Run every song's 10s clip through the identifier and tabulate results.

In [ ]:
correct_count = 0
results_table = []
for song_file in songs:
    name = song_file.replace('.mp3', '')
    path = os.path.join(DB_DIR, song_file)
    y    = load_audio(path, duration=12)
    y_q  = y[SR*1:SR*11]   # 10s clip starting 1s in (avoid intro silence)
    m, sc, _, _, _, _ = identify_clip(y_q, database, song_names)
    ok   = (m == name)
    if ok: correct_count += 1
    results_table.append((name[:35], (m or '?')[:35], 'OK' if ok else 'FAIL'))
    print(f'  {"OK  " if ok else "FAIL"}  {name[:35]:<35} -> {m}')

accuracy = correct_count / len(songs) * 100
print(f'\nAccuracy: {correct_count}/{len(songs)} = {accuracy:.1f}%')

fig, ax = plt.subplots(figsize=(14, len(songs)*0.30+1.5), facecolor=DARK)
ax.set_facecolor(DARK); ax.axis('off')
rows = [['Query Song', 'Matched Song', 'Result']] + results_table
table = ax.table(cellText=rows[1:], colLabels=rows[0], cellLoc='left', loc='center', colWidths=[0.44, 0.44, 0.12])
table.auto_set_font_size(False); table.set_fontsize(8.5)
for (r, c), cell in table.get_celld().items():
    cell.set_facecolor('#1f2937' if r % 2 == 0 else PANEL)
    cell.set_edgecolor('#374151')
    if r == 0: cell.set_facecolor('#1e3a5f')
    txt = cell.get_text().get_text()
    if txt == 'OK': cell.get_text().set_color(GREEN)
    elif txt == 'FAIL': cell.get_text().set_color(RED)
    else: cell.get_text().set_color('white')
ax.set_title(f'Full Database Recognition Accuracy: {correct_count}/{len(songs)} = {accuracy:.1f}%',
             color=BLUE, fontsize=13, fontweight='bold', pad=10)
plt.tight_layout()
savefig(fig, 'q3a_7_accuracy_table')

print(f'\nAll Q3A figures saved to {OUT}/   Accuracy={accuracy:.1f}%')


## Step 17 — Download all figures (optional)

In [ ]:
import shutil
shutil.make_archive('/content/q3a_figures', 'zip', OUT)
from google.colab import files
files.download('/content/q3a_figures.zip')
